# Notebook 6 — Figures & Comparative Analysis

Generates all publication-quality figures from evaluation results.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

METRICS_JSON = '../evaluation/full_metrics.json'
FIGURES_DIR  = '../figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

from scripts.generate_figures import generate_all_figures
generate_all_figures(metrics_json=METRICS_JSON, output_dir=FIGURES_DIR)
print('\n✓ All figures saved to', FIGURES_DIR)

In [ ]:
# Display figures inline
import glob
from IPython.display import Image, display

for fig_path in sorted(glob.glob(f'{FIGURES_DIR}/*.png')):
    print(os.path.basename(fig_path))
    display(Image(filename=fig_path, width=900))
    print()

## Research Questions — Summary

Answer the key research questions based on the evaluation results.

In [ ]:
import json, pandas as pd

with open(METRICS_JSON) as f:
    all_m = json.load(f)

print('=' * 60)
print('RQ1: Which LLM performs best overall?')
print('=' * 60)
for task, rows in all_m.items():
    df = pd.DataFrame(rows).dropna(subset=['f1'])
    # Exclude ensemble for individual model comparison
    individual = df[df['model'] != 'ensemble']
    best = individual.loc[individual['f1'].idxmax()]
    print(f'  {task.upper()}: {best["model"]} ({best["strategy"]})  F1={best["f1"]:.4f}')

print()
print('=' * 60)
print('RQ2: Which prompting strategy is most effective?')
print('=' * 60)
for task, rows in all_m.items():
    df = pd.DataFrame(rows).dropna(subset=['f1'])
    by_strategy = df.groupby('strategy')['f1'].mean().sort_values(ascending=False)
    best_strat = by_strategy.index[0]
    print(f'  {task.upper()}: {best_strat}  (avg F1={by_strategy.iloc[0]:.4f})')

print()
print('=' * 60)
print('RQ3: Does ensemble improve over individual models?')
print('=' * 60)
for task, rows in all_m.items():
    df = pd.DataFrame(rows).dropna(subset=['f1'])
    ens = df[df['model'] == 'ensemble']['f1'].max()
    ind = df[df['model'] != 'ensemble']['f1'].max()
    improved = 'YES ✓' if ens > ind else 'NO ✗'
    print(f'  {task.upper()}: Ensemble F1={ens:.4f} vs Best Individual F1={ind:.4f} → {improved}')

print()
print('=' * 60)
print('RQ4: Which defect type is harder to detect?')
print('=' * 60)
task_best = {}
for task, rows in all_m.items():
    df = pd.DataFrame(rows).dropna(subset=['f1'])
    task_best[task] = df['f1'].max()
harder = min(task_best, key=task_best.get)
print(f'  {harder.upper()} is harder  (best F1={task_best[harder]:.4f} vs other={list(task_best.values())[1-list(task_best.keys()).index(harder)]:.4f})')